Groundwater | Case Study

# Topic 5 : Model Calibration

Dr. Xiang-Zhao Kong & Dr. Beatrice Marti & Louise Noel du Payrat

In [ ]:
# Import libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import flopy
from flopy.utils import HeadFile

from IPython.display import display

# Load local modules
sys.path.append('../SUPPORT_REPO/src')
sys.path.append('../SUPPORT_REPO/src/scripts/scripts_exercises')

from data_utils import (
    download_named_file,
    get_default_data_folder,
)
import print_images as du

# Set up plotting defaults
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

## Introduction

In the previous notebooks, we defined our modeling objectives (Notebook 1), developed a perceptual model of the Limmat Valley aquifer (Notebook 2), chose the conceptual model (Notebook 3), and implemented the numerical model using MODFLOW-NWT (Notebook 4). 

Now we move to **Step 5: Model Calibration** - the process of adjusting model parameters so that simulated outputs match observed measurements. This is a critical step that bridges the gap between our theoretical model and the real-world system.

### What is Calibration?

Calibration is the process of adjusting model parameters within physically reasonable bounds to minimize the difference between simulated and observed values. In groundwater modeling, we typically calibrate against:

- **Hydraulic heads** (groundwater levels) at observation wells
- **Groundwater fluxes** (e.g., baseflow to rivers, spring discharge)
- **Contaminant concentrations** (for transport models)

### Calibration Workflow

1. **Define calibration targets**: Select observation data and define metrics
2. **Run the model**: Execute with initial parameter estimates
3. **Compare outputs**: Calculate residuals (simulated - observed)
4. **Adjust parameters**: Modify parameters to reduce residuals
5. **Iterate**: Repeat until acceptable fit is achieved
6. **Evaluate**: Check physical plausibility and water balance

> **Important: Calibration vs. Validation**
>
> We should **never** use all available data for calibration. A portion of the data must be reserved for independent validation (Step 6). This ensures the model has predictive capability, not just good fit to training data.
>
> Common splits are 75%/25% or 70%/30% for calibration/validation.

## 1 - Loading the Calibrated Model

We start by loading the MODFLOW-NWT model that we built in Notebook 4. The model files are stored in the workspace directory.

In [ ]:
# Define model paths
model_name = 'limmat_valley_model_nwt'
workspace = os.path.join(get_default_data_folder(), 'calibration')

# Check if model directory exists
if not os.path.exists(workspace):
    # Create the directory if it doesn't exist
    os.makedirs(workspace)
    
    # Try to download and extract baseline model
    print("\nAttempting to download baseline model...")
    baseline_path = download_named_file('baseline_model', data_type='calibration')

    # Extract if it's a zip file
    if baseline_path.endswith('.zip'):
        import zipfile
        extract_dir = os.path.dirname(baseline_path)
        with zipfile.ZipFile(baseline_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Extracted to: {extract_dir}")
        
        # Update workspace path
        #workspace = os.path.join(extract_dir, model_name)

else:
    print(f"Model workspace found at: {workspace}")

In [ ]:
# Load the MODFLOW-NWT model
mf = flopy.modflow.Modflow.load(
    f"{model_name}.nam",
    model_ws=workspace,
    exe_name='mfnwt',
    version='mfnwt',
    verbose=False
)

print(f"Model loaded: {mf.name}")
print(f"Number of layers: {mf.nlay}")
print(f"Number of rows: {mf.nrow}")
print(f"Number of columns: {mf.ncol}")
print(f"Packages: {[pkg.name[0] for pkg in mf.packagelist]}")

## 2 - Loading Observation Data

We use groundwater level observations from the AWEL (Amt fur Abfall, Wasser, Energie und Luft) monitoring network, which we introduced in Notebook 2 (Chapter 6: Monitoring the Limmat Valley Aquifer).

### 2.1 - Groundwater Level Time Series

The time series data contains daily groundwater levels from multiple observation wells.

In [ ]:
# Download groundwater time series data
gw_ts_path = download_named_file(
    name='groundwater_timeseries',
    data_type='time_series'
)

# Load the data
gw_ts_raw = pd.read_csv(gw_ts_path)

# Process the data - keep coordinates from the raw data
gw_ts = gw_ts_raw[['date', 'value', 'well_id', 'x_coord', 'y_coord']].copy()
gw_ts['date'] = pd.to_datetime(gw_ts['date'], errors='coerce')
gw_ts = gw_ts.drop_duplicates()
gw_ts = gw_ts.sort_values('date')
# Make sure well_id is string
gw_ts['well_id'] = gw_ts['well_id'].astype(str)

# Drop rows with invalid well_ids (e.g., NaN and 'nan')
gw_ts = gw_ts[~gw_ts['well_id'].isin(['nan', 'NaN', ''])]
# Drop rows with invalid dates
gw_ts = gw_ts.dropna(subset=['date'])

# Standardize well IDs (B53 and B5-3 are the same well)
gw_ts.loc[gw_ts['well_id'] == 'B53', 'well_id'] = 'B5-3'

print(f"Loaded {len(gw_ts)} observations from {gw_ts['well_id'].nunique()} wells")
print(f"Date range: {gw_ts['date'].min()} to {gw_ts['date'].max()}")
print(f"\nWells: {sorted(gw_ts['well_id'].unique())}")

In [ ]:
# Calculate summary statistics for each well
well_stats = gw_ts.groupby('well_id')['value'].agg(
    ['mean', 'std', 'min', 'max', 'count']
).round(2)
well_stats['range'] = well_stats['max'] - well_stats['min']
well_stats = well_stats.sort_values('mean', ascending=False)

print("Summary statistics for observation wells (head in m a.s.l.):")
display(well_stats)

### 2.2 - Observation Well Locations

The groundwater time series data already includes the spatial coordinates (x_coord, y_coord) for each observation well. We can use these directly to find the corresponding model grid cells.

In [ ]:
# Extract unique well locations from the time series data
well_locations = gw_ts.groupby('well_id').agg({
    'x_coord': 'first',
    'y_coord': 'first'
}).reset_index()

# Convert coordinates to numeric (they may be strings)
well_locations['x_coord'] = pd.to_numeric(well_locations['x_coord'], errors='coerce')
well_locations['y_coord'] = pd.to_numeric(well_locations['y_coord'], errors='coerce')

print(f"Found {len(well_locations)} unique observation wells with coordinates:")
display(well_locations)

### 2.3 - Matching Wells to Model Grid

We need to find which model cell each observation well falls into. This allows us to extract simulated heads at the correct locations.

In [ ]:
# First, let's check the model grid properties
print("Model Grid Properties:")
print(f"  X offset (xoffset): {mf.modelgrid.xoffset}")
print(f"  Y offset (yoffset): {mf.modelgrid.yoffset}")
print(f"  Rotation angle (angrot): {mf.modelgrid.angrot} degrees")
print(f"  Grid extent: {mf.modelgrid.extent}")
print(f"  Grid shape: {mf.modelgrid.nrow} rows x {mf.modelgrid.ncol} cols")

# Check well coordinates vs grid extent
print(f"\nWell coordinate ranges:")
print(f"  X: {well_locations['x_coord'].min():.0f} to {well_locations['x_coord'].max():.0f}")
print(f"  Y: {well_locations['y_coord'].min():.0f} to {well_locations['y_coord'].max():.0f}")

In [ ]:
def find_cell_for_point(modelgrid, x, y):
    """
    Find the model cell (row, col) containing a point.
    Handles rotated grids by converting to local coordinates first.
    
    Parameters
    ----------
    modelgrid : flopy.discretization.StructuredGrid
        The model grid object
    x, y : float
        Coordinates of the point in real-world coordinates
        
    Returns
    -------
    tuple or None
        (row, col) if point is within grid, None otherwise
    """
    try:
        # For rotated grids, we need to convert to local coordinates first
        if modelgrid.angrot != 0:
            local_x, local_y = modelgrid.get_local_coords(x, y)
        else:
            local_x, local_y = x, y
        
        # Use FloPy's intersect method (works with real-world coords)
        row, col = modelgrid.intersect(x, y)
        return (int(row), int(col))
    except Exception as e:
        return None

# Test the function with the first well location
test_x = well_locations['x_coord'].iloc[0]
test_y = well_locations['y_coord'].iloc[0]
cell = find_cell_for_point(mf.modelgrid, test_x, test_y)
print(f"Test point ({test_x:.0f}, {test_y:.0f}) -> Cell: {cell}")

In [ ]:
# Find model cells for all observation wells
obs_wells = []
wells_outside = []
wells_inactive = []

for idx, row in well_locations.iterrows():
    well_id = row['well_id']
    x, y = row['x_coord'], row['y_coord']
    
    # Skip wells with missing coordinates
    if pd.isna(x) or pd.isna(y):
        print(f"Skipping well {well_id}: missing coordinates")
        continue
    
    cell = find_cell_for_point(mf.modelgrid, x, y)
    if cell is not None:
        i, j = cell
        # Check if cell is active (ibound > 0)
        if hasattr(mf, 'bas6') and mf.bas6.ibound.array[0, i, j] > 0:
            obs_wells.append({
                'well_id': well_id,
                'x': x,
                'y': y,
                'row': i,
                'col': j,
                'layer': 0
            })
        else:
            wells_inactive.append(well_id)
    else:
        wells_outside.append(well_id)

obs_wells_df = pd.DataFrame(obs_wells)

print(f"Results:")
print(f"  Wells within active model domain: {len(obs_wells_df)}")
print(f"  Wells outside model domain: {len(wells_outside)} - {wells_outside}")
print(f"  Wells in inactive cells: {len(wells_inactive)} - {wells_inactive}")

if len(obs_wells_df) > 0:
    display(obs_wells_df)
else:
    print("\nWARNING: No observation wells found within the active model domain!")
    print("Check the visualization in the next cell to diagnose the issue.")

In [ ]:
# Visualize well locations relative to model grid
fig, ax = plt.subplots(figsize=(12, 10))

# Plot the model grid with ibound
pmv = flopy.plot.PlotMapView(model=mf, ax=ax)
ibound_plot = pmv.plot_ibound()
pmv.plot_grid(linewidth=0.3, alpha=0.5)

# Plot all well locations
ax.scatter(well_locations['x_coord'], well_locations['y_coord'], 
           c='red', s=100, marker='o', label='Observation wells', zorder=5, edgecolors='black')

# Add well labels
for idx, row in well_locations.iterrows():
    ax.annotate(row['well_id'], (row['x_coord'], row['y_coord']), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Observation Well Locations vs Model Grid')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate mean heads for calibration
# We use a recent period (e.g., 2015-2020) to match current conditions
calibration_period = (gw_ts['date'] >= '2015-01-01') & (gw_ts['date'] <= '2020-12-31')
gw_ts_calib = gw_ts[calibration_period].copy()

# Calculate mean head per well
mean_heads = gw_ts_calib.groupby('well_id')['value'].mean().round(2)
mean_heads.name = 'observed_head'

print(f"Mean observed heads (2015-2020):")
print(mean_heads)

## 3 - Calibration Metrics

We define several metrics to quantify the fit between simulated and observed heads.

In [ ]:
def calculate_calibration_metrics(observed, simulated):
    """
    Calculate calibration performance metrics.
    
    Parameters
    ----------
    observed : array-like
        Observed values
    simulated : array-like
        Simulated values
        
    Returns
    -------
    dict
        Dictionary of metrics
    """
    obs = np.array(observed)
    sim = np.array(simulated)
    
    # Remove any NaN values
    mask = ~(np.isnan(obs) | np.isnan(sim))
    obs = obs[mask]
    sim = sim[mask]
    
    n = len(obs)
    if n == 0:
        return None
    
    residuals = sim - obs
    
    # Mean Error (Bias)
    me = np.mean(residuals)
    
    # Mean Absolute Error
    mae = np.mean(np.abs(residuals))
    
    # Root Mean Square Error
    rmse = np.sqrt(np.mean(residuals**2))
    
    # Coefficient of determination (R^2)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((obs - np.mean(obs))**2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    # Normalized RMSE (as percentage of observed range)
    obs_range = obs.max() - obs.min()
    nrmse = (rmse / obs_range * 100) if obs_range > 0 else 0
    
    return {
        'n': n,
        'ME (bias)': round(me, 3),
        'MAE': round(mae, 3),
        'RMSE': round(rmse, 3),
        'NRMSE (%)': round(nrmse, 1),
        'R2': round(r2, 3)
    }

# Test with dummy data
test_obs = [100, 101, 102, 103]
test_sim = [100.5, 101.2, 101.8, 103.1]
print("Test metrics:", calculate_calibration_metrics(test_obs, test_sim))

### Interpretation of Metrics

| Metric | Description | Good Value |
|--------|-------------|------------|
| **ME (Bias)** | Mean error - positive means model over-predicts | Close to 0 |
| **MAE** | Mean absolute error | < 1-2 m for regional models |
| **RMSE** | Root mean square error - sensitive to outliers | < 1-2 m for regional models |
| **NRMSE** | Normalized RMSE as % of observed range | < 10% is good |
| **R^2** | Coefficient of determination | > 0.9 is good |

## 4 - Initial Model Run

Let's run the model with the initial (uncalibrated) parameters and compare against observations.

In [ ]:
# Write input files and run the model
mf.write_input()
success, buff = mf.run_model(silent=True)

if success:
    print("Model run completed successfully!")
else:
    print("Model run failed. Check the listing file for errors.")
    # Print last few lines of buffer for debugging
    print("\nLast lines of output:")
    for line in buff[-10:]:
        print(line)

In [ ]:
# Load simulated heads
head_file = os.path.join(workspace, f"{model_name}.hds")

if os.path.exists(head_file):
    hds = HeadFile(head_file)
    head = hds.get_data(kstpkper=(0, 0))  # Steady state: first time step, first stress period
    print(f"Head array shape: {head.shape}")
    print(f"Head range: {np.nanmin(head):.2f} to {np.nanmax(head):.2f} m")
else:
    print(f"Head file not found: {head_file}")

In [ ]:
# Extract simulated heads at observation well locations
def extract_simulated_heads(head_array, obs_wells_df, layer=0):
    """
    Extract simulated heads at observation well locations.
    
    Parameters
    ----------
    head_array : np.ndarray
        3D array of simulated heads [nlay, nrow, ncol]
    obs_wells_df : pd.DataFrame
        DataFrame with observation well locations (row, col columns)
    layer : int
        Layer index to extract from
        
    Returns
    -------
    np.ndarray
        Array of simulated heads at observation locations
    """
    sim_heads = []
    for _, well in obs_wells_df.iterrows():
        i, j = int(well['row']), int(well['col'])
        h = head_array[layer, i, j]
        # Check for dry cells (MODFLOW uses large negative values)
        if h < -999:
            h = np.nan
        sim_heads.append(h)
    return np.array(sim_heads)

# Extract simulated heads (only if we have observation wells)
if 'head' in dir() and 'obs_wells_df' in dir() and len(obs_wells_df) > 0:
    obs_wells_df['simulated_head'] = extract_simulated_heads(head, obs_wells_df)
    print("Simulated heads extracted at observation wells:")
    display(obs_wells_df)
else:
    if 'obs_wells_df' not in dir() or len(obs_wells_df) == 0:
        print("No observation wells found in active model domain - cannot extract simulated heads.")
        print("Please check the well locations and model grid alignment.")
    elif 'head' not in dir():
        print("Head array not available - run the model first.")

## 5 - Calibration Assessment

Now let's compare simulated vs. observed heads and calculate performance metrics.

In [ ]:
# Create comparison DataFrame by matching well IDs between obs_wells_df and mean_heads
if 'obs_wells_df' in dir() and len(obs_wells_df) > 0:
    comparison_df = obs_wells_df.copy()
    
    # Match observed heads from mean_heads using well_id
    comparison_df['observed_head'] = comparison_df['well_id'].map(mean_heads)
    
    # Calculate residuals (simulated - observed)
    comparison_df['residual'] = comparison_df['simulated_head'] - comparison_df['observed_head']
    
    print("Comparison of simulated vs observed heads:")
    display(comparison_df)
    
    # Show matching statistics
    n_matched = comparison_df['observed_head'].notna().sum()
    n_total = len(comparison_df)
    print(f"\nMatched {n_matched} of {n_total} observation wells with mean head data")
    
    if n_matched > 0:
        # Calculate and display calibration metrics
        valid_mask = comparison_df['observed_head'].notna() & comparison_df['simulated_head'].notna()
        if valid_mask.any():
            metrics = calculate_calibration_metrics(
                comparison_df.loc[valid_mask, 'observed_head'].values,
                comparison_df.loc[valid_mask, 'simulated_head'].values
            )
            print(f"\nCalibration Metrics:")
            for key, value in metrics.items():
                print(f"  {key}: {value}")
else:
    print("No observation wells available for comparison.")

In [ ]:
def plot_calibration_scatter(observed, simulated, title="Calibration Results"):
    """
    Create a scatter plot of simulated vs observed heads.
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Remove NaN values
    mask = ~(np.isnan(observed) | np.isnan(simulated))
    obs = np.array(observed)[mask]
    sim = np.array(simulated)[mask]
    
    if len(obs) == 0:
        print("No valid data points for plotting")
        return fig, ax
    
    # Plot points
    ax.scatter(obs, sim, s=100, alpha=0.7, edgecolors='black', linewidth=1)
    
    # Add 1:1 line
    min_val = min(obs.min(), sim.min())
    max_val = max(obs.max(), sim.max())
    margin = (max_val - min_val) * 0.05
    line_range = [min_val - margin, max_val + margin]
    ax.plot(line_range, line_range, 'k--', label='1:1 line', linewidth=2)
    
    # Add +/- 1m error bands
    ax.fill_between(line_range, 
                    [x - 1 for x in line_range], 
                    [x + 1 for x in line_range],
                    alpha=0.2, color='gray', label='+/- 1m')
    
    # Calculate and display metrics
    metrics = calculate_calibration_metrics(obs, sim)
    if metrics:
        text = f"n = {metrics['n']}\n"
        text += f"RMSE = {metrics['RMSE']:.2f} m\n"
        text += f"MAE = {metrics['MAE']:.2f} m\n"
        text += f"R² = {metrics['R2']:.3f}\n"
        text += f"Bias = {metrics['ME (bias)']:.2f} m"
        ax.text(0.05, 0.95, text, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Observed Head (m a.s.l.)', fontsize=12)
    ax.set_ylabel('Simulated Head (m a.s.l.)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(loc='lower right')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig, ax

# Plot calibration results (if data available)
if 'comparison_df' in dir() and not comparison_df['observed_head'].isna().all():
    fig, ax = plot_calibration_scatter(
        comparison_df['observed_head'].values,
        comparison_df['simulated_head'].values,
        title="Figure 1: Initial Calibration - Simulated vs Observed Heads"
    )
    plt.show()
else:
    print("Observed head data not yet matched. See TODO above.")

In [ ]:
def plot_residual_map(modelgrid, obs_wells_df, head_array, title="Residual Map"):
    """
    Create a map showing spatial distribution of residuals.
    """
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Plot head distribution
    pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, ax=ax)
    head_plot = pmv.plot_array(head_array[0], cmap='viridis', alpha=0.7)
    plt.colorbar(head_plot, ax=ax, label='Simulated Head (m a.s.l.)', shrink=0.7)
    
    # Plot observation wells colored by residual
    if 'residual' in obs_wells_df.columns:
        valid_mask = ~obs_wells_df['residual'].isna()
        if valid_mask.any():
            scatter = ax.scatter(
                obs_wells_df.loc[valid_mask, 'x'],
                obs_wells_df.loc[valid_mask, 'y'],
                c=obs_wells_df.loc[valid_mask, 'residual'],
                cmap='RdBu_r',
                s=200,
                edgecolors='black',
                linewidth=2,
                vmin=-3, vmax=3,
                zorder=5
            )
            plt.colorbar(scatter, ax=ax, label='Residual (m)', shrink=0.5)
    
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title(title)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    return fig, ax

# Plot residual map
if 'head' in dir() and 'obs_wells_df' in dir():
    fig, ax = plot_residual_map(
        mf.modelgrid, 
        obs_wells_df, 
        head,
        title="Figure 2: Spatial Distribution of Calibration Residuals"
    )
    plt.show()

## 6 - Manual Calibration

Based on the residual analysis, we can identify which parameters need adjustment. Common calibration parameters include:

1. **Hydraulic conductivity (K)** - affects head gradients and flow velocities
2. **Recharge rates** - affects overall water balance
3. **River bed conductance** - affects river-aquifer exchange
4. **Boundary condition heads** - affects heads near boundaries

### 6.1 - Parameter Sensitivity

Before adjusting parameters, let's understand which parameters have the most influence.

In [ ]:
# Get current parameter values
print("Current model parameters:")
print("=" * 50)

# Hydraulic conductivity (UPW or LPF package)
if hasattr(mf, 'upw'):
    hk = mf.upw.hk.array
    print(f"\nHydraulic Conductivity (K):")
    print(f"  Min: {np.nanmin(hk):.2e} m/day")
    print(f"  Max: {np.nanmax(hk):.2e} m/day")
    print(f"  Mean: {np.nanmean(hk):.2e} m/day")
elif hasattr(mf, 'lpf'):
    hk = mf.lpf.hk.array
    print(f"\nHydraulic Conductivity (K):")
    print(f"  Min: {np.nanmin(hk):.2e} m/day")
    print(f"  Max: {np.nanmax(hk):.2e} m/day")
    print(f"  Mean: {np.nanmean(hk):.2e} m/day")

# Recharge
if hasattr(mf, 'rch'):
    rch = mf.rch.rech.array
    print(f"\nRecharge:")
    print(f"  Mean: {np.nanmean(rch)*1000:.2f} mm/day")
    print(f"  Annual: {np.nanmean(rch)*365*1000:.0f} mm/year")

# River package
if hasattr(mf, 'riv'):
    print(f"\nRiver cells: {len(mf.riv.stress_period_data[0])}")

### 6.2 - Parameter Adjustment Strategy

Based on the residual patterns:

- **Systematic bias (all residuals positive/negative)**: Adjust recharge or boundary heads
- **Spatial gradient issues**: Adjust hydraulic conductivity or its spatial distribution
- **Local errors near rivers**: Adjust river bed conductance

In [ ]:
# Example: Adjust hydraulic conductivity by a factor
def run_calibration_trial(mf, k_factor=1.0, rch_factor=1.0):
    """
    Run a calibration trial with adjusted parameters.
    
    Parameters
    ----------
    mf : flopy.modflow.Modflow
        The model object
    k_factor : float
        Multiplier for hydraulic conductivity
    rch_factor : float
        Multiplier for recharge
        
    Returns
    -------
    np.ndarray or None
        Head array if successful, None if failed
    """
    import copy
    
    # Create a copy to avoid modifying original
    # Note: This is simplified - full implementation would create new model
    
    # Adjust K
    if hasattr(mf, 'upw'):
        original_hk = mf.upw.hk.array.copy()
        mf.upw.hk = original_hk * k_factor
    elif hasattr(mf, 'lpf'):
        original_hk = mf.lpf.hk.array.copy()
        mf.lpf.hk = original_hk * k_factor
    
    # Adjust recharge
    if hasattr(mf, 'rch'):
        original_rch = mf.rch.rech.array.copy()
        mf.rch.rech = original_rch * rch_factor
    
    # Run model
    mf.write_input()
    success, _ = mf.run_model(silent=True)
    
    # Restore original values
    if hasattr(mf, 'upw'):
        mf.upw.hk = original_hk
    elif hasattr(mf, 'lpf'):
        mf.lpf.hk = original_hk
    if hasattr(mf, 'rch'):
        mf.rch.rech = original_rch
    
    if success:
        hds = HeadFile(os.path.join(mf.model_ws, f"{mf.name}.hds"))
        return hds.get_data(kstpkper=(0, 0))
    return None

print("Calibration trial function defined.")
print("Use run_calibration_trial(mf, k_factor=1.5) to test with 50% higher K.")

## 7 - Calibration Results Summary

After completing the calibration process, summarize the results.

In [ ]:
# Summary table of calibration iterations
# This would be filled in during actual calibration
calibration_log = pd.DataFrame({
    'Iteration': [1],
    'K_factor': [1.0],
    'Rch_factor': [1.0],
    'RMSE (m)': [np.nan],
    'MAE (m)': [np.nan],
    'Bias (m)': [np.nan],
    'R2': [np.nan],
    'Notes': ['Initial run']
})

print("Calibration Log:")
display(calibration_log)

## 8 - Water Balance Check

A calibrated model should have a reasonable water balance. Let's check the model's water budget.

In [ ]:
# Read the cell budget file
cbc_file = os.path.join(workspace, f"{model_name}.cbc")

if os.path.exists(cbc_file):
    cbc = flopy.utils.CellBudgetFile(cbc_file)
    
    print("Available budget terms:")
    print(cbc.get_unique_record_names())
    
    # Get budget summary
    print("\n" + "="*50)
    print("WATER BALANCE SUMMARY")
    print("="*50)
    
    for term in cbc.get_unique_record_names():
        data = cbc.get_data(text=term, kstpkper=(0,0))
        if data:
            total = np.sum(data[0])
            print(f"{term.strip():20s}: {total:>15.2f} m³/day")
else:
    print(f"Budget file not found: {cbc_file}")

## 9 - Discussion

### Key Takeaways

1. **Calibration is iterative**: Multiple rounds of parameter adjustment are typically needed
2. **Physical plausibility matters**: Parameters must stay within realistic bounds
3. **Non-uniqueness**: Different parameter combinations can produce similar fits
4. **Trade-offs**: Improving fit at one location may worsen it elsewhere

### Limitations of Manual Calibration

- Time-consuming for many parameters
- Difficult to explore full parameter space
- Subjective decisions about which parameters to adjust

### When to Use Automated Calibration

For complex models with many parameters, consider tools like:
- **PEST** (Parameter ESTimation): Industry standard for groundwater model calibration
- **pyEMU**: Python interface to PEST, useful for uncertainty analysis

These tools systematically search the parameter space and provide uncertainty estimates.

## Next Steps

In the next notebook (6_validation.ipynb), we will:
1. Test the calibrated model against reserved validation data
2. Assess the model's predictive capability
3. Discuss when recalibration might be needed

## References

1. Anderson, M.P., Woessner, W.W., Hunt, R.J. (2015). Applied Groundwater Modeling: Simulation of Flow and Advective Transport (2nd ed.). Academic Press.

2. Doherty, J. (2015). Calibration and Uncertainty Analysis for Complex Environmental Models. Watermark Numerical Computing.

3. Hill, M.C., Tiedeman, C.R. (2007). Effective Groundwater Model Calibration. Wiley.